# Onboarding 02 ? GenBio-PathFM embeddings

A pretrained image model converts each 224?224 H&E spot patch into a 4,608-number embedding. The model is frozen: this step extracts image features and does not train it. A one-patch demonstration comes before the all-section loop.

Use the same `HIST2ST_DATA_DIR` as Notebook 01. GPU is recommended for the full run; CPU can run the single-patch demonstration. Hugging Face access is needed for the model. Longer notes: [study materials](../../docs/study_materials.md).


## Install this notebook?s model dependencies
Install the model library and image utilities. Install PyTorch using the CPU/GPU instructions in `README-new.md` so a CUDA runtime is not accidentally replaced.

In [ ]:
%pip install -q transformers huggingface_hub pillow


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import h5py
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoModel
from huggingface_hub import login

**Storage:** Mount Drive in Colab when using Drive. For local/Drive Desktop, use the same `HIST2ST_DATA_DIR` used by Notebook 01.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local runtime: no Drive mount needed.')

## Settings

- `TEAM_FOLDER`, `DATA_DIR`: must match Notebook 1
- `BATCH_SIZE`: patches sent to the GPU at once. 64 works on a T4; lower it if you get an out-of-memory error
- `USE_FP16`: half precision on the GPU, about twice as fast with practically the same embeddings

In [ ]:
from pathlib import Path
DATA_DIR = os.environ.get('HIST2ST_DATA_DIR', './data/HEST_breast_ST')
MODEL_NAME = 'genbio-ai/genbio-pathfm'
BATCH_SIZE = 64
USE_FP16 = True


In [ ]:
DATA_DIR = str(Path(DATA_DIR).expanduser().resolve())
SPOT_TABLE_PATH = os.path.join(DATA_DIR, 'spot_table.csv')
if not os.path.exists(SPOT_TABLE_PATH): raise FileNotFoundError('Run Notebook 01 first: '+SPOT_TABLE_PATH)
PATCHES_DIR = os.path.join(DATA_DIR, 'raw', 'patches')
EMBEDDING_DIR = os.path.join(DATA_DIR, 'embeddings')
os.makedirs(EMBEDDING_DIR, exist_ok=True)
print('Reading from:', DATA_DIR, '\nSaving to:', EMBEDDING_DIR)


**Hugging Face login:** paste your token into the box that appears. Never write it in the code or upload it to GitHub.

In [ ]:
login()

## Step 1. Which sections still need embeddings?

We read the spot table from Notebook 1 and check which sections already have an embedding file. Only the missing ones are embedded.

In [ ]:
spot_table = pd.read_csv(SPOT_TABLE_PATH)
sample_ids = list(spot_table["sample_id"].unique())

done = []
to_do = []
for sample_id in sample_ids:
    if os.path.exists(os.path.join(EMBEDDING_DIR, f"{sample_id}.npz")):
        done.append(sample_id)
    else:
        to_do.append(sample_id)

print("Sections in spot table:", len(sample_ids))
print("Already embedded:", len(done))
print("To embed now:", len(to_do), to_do)
print("Spots to embed now:", spot_table["sample_id"].isin(to_do).sum())

## Step 2. Load GenBio-PathFM

Same model and image preparation as our earlier notebooks:
- `Resize((224, 224))`: the patches are already 224×224, so this changes nothing but guards against odd sizes
- `ToTensor()`: pixel values 0–255 → 0–1
- `Normalize(...)`: GenBio-PathFM's own colour mean and SD, so the images look like what the model was trained on

`model.eval()` and `torch.inference_mode()` mean we only use the model; it is never trained.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("No GPU found: this will be slow. Runtime → Change runtime type → T4 GPU.")
print("Device:", device)

model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.697, 0.575, 0.728), std=(0.188, 0.240, 0.187)),
])

### One-patch demonstration
Run one image through the same transform and frozen model used below. This makes the input and embedding shapes visible before processing every spot.

In [ ]:
demo = spot_table.iloc[0]
with h5py.File(os.path.join(PATCHES_DIR, f'{demo.sample_id}.h5'), 'r') as f:
    demo_img = Image.fromarray(f['img'][int(demo.patch_index)]).convert('RGB')
demo_tensor = transform(demo_img).unsqueeze(0).to(device)
with torch.inference_mode():
    demo_output = model(demo_tensor)
if isinstance(demo_output, (tuple, list)): demo_output = demo_output[0]
if hasattr(demo_output, 'pooler_output') and demo_output.pooler_output is not None: demo_output = demo_output.pooler_output
if demo_output.ndim > 2: demo_output = demo_output.flatten(1)
print('Image tensor:', tuple(demo_tensor.shape), '| embedding:', tuple(demo_output.shape))


## Step 3. Embed each section

For each section:
1. Take its spots from the spot table, in `row_in_sample` order (the same order as its expression file).
2. Read only those patches from the `.h5` file, using `patch_index`.
3. Check that the barcodes in the patch file match the spot table, so a row can never be paired with the wrong spot.
4. Send the patches through the model in batches and collect the 4,608 numbers per spot.
5. Save the section's file right away.

In [ ]:
def barcode_to_text(barcode):
    if hasattr(barcode, "ravel"):
        barcode = barcode.ravel()[0]
    if isinstance(barcode, bytes):
        barcode = barcode.decode("utf-8")
    return str(barcode)


def embed_patches(images):
    batch = [transform(Image.fromarray(image).convert("RGB")) for image in images]
    x = torch.stack(batch).to(device)
    with torch.inference_mode():
        if USE_FP16 and device.type == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                output = model(x)
        else:
            output = model(x)
    return output.float().cpu().numpy()

In [ ]:
def embed_section(sample_id):
    section_spots = spot_table[spot_table["sample_id"] == sample_id].sort_values("row_in_sample")
    patch_indices = section_spots["patch_index"].to_numpy()
    assert np.all(np.diff(patch_indices) > 0), "patch_index must be increasing"

    patches_path = os.path.join(PATCHES_DIR, f"{sample_id}.h5")
    section_embeddings = []

    with h5py.File(patches_path, "r") as f:
        file_barcodes = f["barcode"][patch_indices]
        file_barcodes = np.array([barcode_to_text(b) for b in file_barcodes])
        assert np.array_equal(file_barcodes, section_spots["barcode"].to_numpy()), \
            f"{sample_id}: barcodes in the patch file don't match the spot table"

        for start in range(0, len(patch_indices), BATCH_SIZE):
            batch_indices = patch_indices[start:start + BATCH_SIZE]
            images = f["img"][batch_indices]
            section_embeddings.append(embed_patches(images))

    section_embeddings = np.concatenate(section_embeddings, axis=0)

    np.savez(
        os.path.join(EMBEDDING_DIR, f"{sample_id}.npz"),
        embeddings=section_embeddings,
        barcode=section_spots["barcode"].to_numpy(),
        sample_id=section_spots["sample_id"].to_numpy(),
        patient=section_spots["patient"].to_numpy(),
    )
    return section_embeddings.shape

In [ ]:
start_time = time.time()

for number, sample_id in enumerate(to_do, start=1):
    shape = embed_section(sample_id)
    minutes = (time.time() - start_time) / 60
    print(f"[{number}/{len(to_do)}] {sample_id}: {shape[0]} spots x {shape[1]} numbers  ({minutes:.1f} min so far)", flush=True)

print("All sections embedded.")

## Step 4. Check the result

Every section in the spot table should now have an embedding file with:
- one row per spot, in the same order as the spot table (barcodes match)
- 4,608 numbers per row
- no missing values (NaN)

In [ ]:
check_rows = []

for sample_id in sample_ids:
    saved = np.load(os.path.join(EMBEDDING_DIR, f"{sample_id}.npz"), allow_pickle=True)
    embeddings = saved["embeddings"]
    section_spots = spot_table[spot_table["sample_id"] == sample_id].sort_values("row_in_sample")

    check_rows.append({
        "sample_id": sample_id,
        "patient": section_spots["patient"].iloc[0],
        "spots": embeddings.shape[0],
        "numbers_per_spot": embeddings.shape[1],
        "barcodes_match": np.array_equal(saved["barcode"], section_spots["barcode"].to_numpy()),
        "any_nan": bool(np.isnan(embeddings).any()),
    })

check_table = pd.DataFrame(check_rows)
print(check_table.to_string(index=False))
print()
print("Total spots embedded:", check_table["spots"].sum(), "of", len(spot_table))
print("All barcodes match:", check_table["barcodes_match"].all())
print("Any NaN:", check_table["any_nan"].any())

## Next

Run **Notebook 3 (PCA + Ridge)**. It needs no GPU: switch back to a **CPU** runtime to save your GPU time.